In [1]:
import sys
!{sys.executable} -m pip install -q torch transformers scikit-learn scipy numpy

In [2]:
import torch
from transformers import BertTokenizer, BertModel
import numpy as np
from scipy.spatial.distance import cosine

# --- 1. Define Sample Sentences ---
# We use the polysemous word "bank" to test context sensitivity
s1 = "I went to the river bank to fish."
s2 = "I went to the savings bank to deposit money."
s3 = "The river flow was strong at the bank."

# --- 2. Word2Vec (Simulated Static Behavior) ---
# Static embeddings assign one fixed vector per word.
# We'll simulate this by assigning a random vector to 'bank' 
# that stays the same regardless of context.
np.random.seed(42)
static_bank_vector = np.random.rand(768) 

# --- 3. BERT (Contextual Embeddings) ---
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

def get_bert_word_vector(sentence, target_word):
    inputs = tokenizer(sentence, return_tensors="pt")
    outputs = model(**inputs)
    
    # Get token index of the target word
    token_list = tokenizer.tokenize(sentence)
    # Note: BERT adds [CLS] at index 0, so we add 1 to the word index
    target_idx = token_list.index(target_word.lower()) + 1
    
    # Extract the vector from the last hidden layer
    return outputs.last_hidden_state[0, target_idx, :].detach().numpy()

# Get BERT vectors for 'bank' in different contexts
bert_v1 = get_bert_word_vector(s1, "bank") # River context
bert_v2 = get_bert_word_vector(s2, "bank") # Finance context
bert_v3 = get_bert_word_vector(s3, "bank") # Another river context

# --- 4. Comparison Logic ---
def similarity(v1, v2):
    return 1 - cosine(v1, v2)

print("="*60)
print("COMPARING EMBEDDING BEHAVIOR")
print("="*60)

# Static Analysis
print(f"STATIC (Word2Vec) Similarity [S1 vs S2]: {similarity(static_bank_vector, static_bank_vector):.4f}")
print("Observation: Static models always return 1.0 similarity for the same word.\n")

# Contextual Analysis
sim_12 = similarity(bert_v1, bert_v2) # River vs Finance
sim_13 = similarity(bert_v1, bert_v3) # River vs River

print(f"BERT Similarity [River vs Finance]: {sim_12:.4f}")
print(f"BERT Similarity [River vs River]:   {sim_13:.4f}")

if sim_13 > sim_12:
    print("\nResult: BERT successfully identified that 'bank' in S1 is more similar to S3 than S2!")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

C:\Users\Manasi\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Manasi\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


COMPARING EMBEDDING BEHAVIOR
STATIC (Word2Vec) Similarity [S1 vs S2]: 1.0000
Observation: Static models always return 1.0 similarity for the same word.

BERT Similarity [River vs Finance]: 0.6163
BERT Similarity [River vs River]:   0.7263

Result: BERT successfully identified that 'bank' in S1 is more similar to S3 than S2!
